# Section 4: Testing Portability on Sectoral Downturns

**Research question:** Does skill/geographic portability predict the degree of labor displacement when an occupation experiences a demand downturn?

**Model (Equation 7):**

$$\text{WeeksUnemployed}_{o,y} = \alpha_0 + \alpha_1 \cdot \text{PostingsChange}_{o,y} + \alpha_2 \cdot \text{Portability}_o + u_{o,y}$$

- $\alpha_2$ is Key Evaluation Metric 2: the predictive role of portability in labor displacement
- **Weeks unemployed** — from CPS ASEC, workers in occupation $o$ reporting weeks unemployed last year, interview month $m$, year $y$
- **Postings change** — 12-month % change in job postings (Lightcast) ending in year $y$, averaged across months
- **Portability** — occupation-level portability index (Jacob's measure)

**Sample:** 2020–2025. Requires 2019 postings data to compute 12-month changes for Jan 2020.

**Crosswalk chain:**
- Job postings → SOC5 (Lightcast 2021) → OCC2010 (Lightcast crosswalk)
- CPS weeks unemployed → OCC (Census 2018, comparable to OCC2010 for most codes)
- Portability → OCC (Census codes, same as OCC2010)
- Merge key: **OCC2010 × Year**

In [1]:
import pandas as pd
import numpy as np
import os
import glob
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import snowflake.connector as snow

pd.set_option('display.max_columns', None)

user     = os.environ['USER']
username = os.environ['SNOWFLAKE_USER']
sf_pass  = os.environ['SNOWFLAKE_PASSWORD']

data_path = f"/Users/{user}/Documents/USF/ECON696/updated code/data/section 4/"

conn = snow.connect(
    user=username,
    password=sf_pass,
    account='avb99459.us-east-1',
    warehouse='OPPORTUNITYATWORK_WH',
    role='PUBLIC'
)
cur = conn.cursor()

---
## Step 1: Load CPS ASEC — Weeks Unemployed

Source: IPUMS CPS extract (`cps_00064.csv`). Key variables:
- **`OCCLY`** — Census occupation code **last year** ← use this, not `OCC`
- **`OCC`** — Census occupation code at the time of the survey (current)
- `WKSUNEM1` — weeks unemployed last year; coded 99 = not in universe (set to NaN)
- `ASECWT` — ASEC person weight (used for weighted means)
- `ASECFLAG` — filter to 1 (ASEC supplement respondents only)

**Why `OCCLY` and not `OCC`:** Both `WKSUNEM1` and `OCCLY` refer to the same reference period — the prior calendar year. Using `OCC` (current occupation) would misattribute unemployment weeks to the wrong occupation for anyone who switched jobs because of a downturn — exactly the displacement dynamic we're trying to measure.

**Sample:** Filtered to labor force participants — `EMPSTAT` ∈ {10, 12, 21, 22}: currently employed (at work or absent) and currently unemployed (experienced and new workers). NILF (30–36), NIU (0), and Armed Forces (1) are excluded. This broader sample captures both workers who experienced unemployment spells during the year and are now re-employed, and workers who remain unemployed at survey time.

The ASEC is fielded in March each year. Respondents report their experience in the *prior* calendar year, so YEAR=2021 → weeks unemployed during 2020. We match this to job postings changes in the same survey year.

In [2]:
cps_raw = pd.read_csv(data_path + 'cps_00064.csv')
cps_raw.columns = cps_raw.columns.str.lower()
print(f'Raw rows: {len(cps_raw):,}')
print(f'Years:    {sorted(cps_raw["year"].unique())}')
cps_raw.head(3)

Raw rows: 906,757
Years:    [2020, 2021, 2022, 2023, 2024, 2025]


,year,serial,month,cpsid,asecflag,asecwth,statefip,pernum,cpsidp,cpsidv,asecwt,age,sex,marst,empstat,occ,occly,wkswork1,wksunem1,migsta1
0,2020,1,3,20190302844900,1,1560.3756,23,1,20190302844901,201903028449011,1560.3756,63,2,1,10,440,440,52,99,99
1,2020,1,3,20190302844900,1,1560.3756,23,2,20190302844902,201903028449021,1560.3756,67,1,1,36,0,8620,40,0,99
2,2020,2,3,20181202843500,1,986.5948,23,1,20181202843501,201812028435011,986.5948,64,1,1,10,9121,9121,52,99,99


In [3]:
# unique EMPSTAT values in the extract
print('EMPSTAT value counts:')
print(cps_raw['empstat'].value_counts().sort_index())
print()
empstat_labels = {
    10: 'At work',
    12: 'Has job, not at work last week',
    20: 'Unemployed',
    21: 'Unemployed, experienced worker',
    22: 'Unemployed, new worker',
    30: 'Not in labor force',
}
print('Labels:')
for v in sorted(cps_raw['empstat'].unique()):
    print(f'  {v}: {empstat_labels.get(v, "other")}')

EMPSTAT value counts:
empstat
0     184790
1       3099
10    407060
12     14796
21     17724
22      1347
32     33702
34    110102
36    134137
Name: count, dtype: int64

Labels:
  0: other
  1: other
  10: At work
  12: Has job, not at work last week
  21: Unemployed, experienced worker
  22: Unemployed, new worker
  32: other
  34: other
  36: other


In [5]:
# filter to ASEC respondents, labor force participants, and recode WKSUNEM1
# empstat: 10=At work, 12=Has job not at work, 21=Unemployed experienced, 22=Unemployed new
cps = (
    cps_raw[
        (cps_raw['asecflag'] == 1) &
        (cps_raw['empstat'].isin([10, 12, 21, 22]))
    ]
    .copy()
    .assign(
        wks_unem=lambda x: x['wksunem1'].replace(99, np.nan)
    )
)

print(f'ASEC respondents (empstat 10/12/21/22): {len(cps):,}')
print(f'EMPSTAT breakdown:')
print(cps['empstat'].value_counts().sort_index())
print()
print(f'WKSUNEM1 = 99 (NIU):  {(cps["wksunem1"]==99).sum():,}')
print(f'Non-missing weeks unemployed: {cps["wks_unem"].notna().sum():,}')
print()
print('Distribution of weeks unemployed (non-zero, non-NIU):')
print(cps[cps['wks_unem'] > 0]['wks_unem'].describe().round(1))


ASEC respondents (empstat 10/12/21/22): 440,927
EMPSTAT breakdown:
empstat
10    407060
12     14796
21     17724
22      1347
Name: count, dtype: int64

WKSUNEM1 = 99 (NIU):  356,723
Non-missing weeks unemployed: 84,204

Distribution of weeks unemployed (non-zero, non-NIU):
count    29792.0
mean        18.0
std         13.4
min          1.0
25%          7.0
50%         14.0
75%         26.0
max         51.0
Name: wks_unem, dtype: float64


In [23]:
# aggregate to occupation x year: weighted mean weeks unemployed
# use OCCLY (occupation last year) — same reference period as WKSUNEM1
cps_occ = (
    cps[cps['wks_unem'].notna() & (cps['occly'] > 0)]
    .groupby(['occly', 'year'])
    .apply(lambda g: np.average(g['wks_unem'], weights=g['asecwt']))
    .reset_index(name='wks_unem_mean')
    .rename(columns={'occly': 'occ'})
)

print(f'Occupation x year cells: {len(cps_occ):,}')
print(f'Unique occupations: {cps_occ["occ"].nunique()}')
print(f'Years: {sorted(cps_occ["year"].unique())}')
cps_occ.head(10)

Occupation x year cells: 2,851
Unique occupations: 525
Years: [2020, 2021, 2022, 2023, 2024, 2025]


/var/folders/6h/l1_lq8b96y72mkslk3phrlpw0000gn/T/ipykernel_62024/3834913316.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: np.average(g['wks_unem'], weights=g['asecwt']))


,occ,year,wks_unem_mean
0,10,2020,3.011996
1,10,2021,5.343740
2,10,2022,4.640960
3,10,2023,2.032711
4,10,2024,2.919432
5,10,2025,6.293947
6,20,2020,4.255933
7,20,2021,8.337944
8,20,2022,9.583706
9,20,2023,4.242057


---
## Step 2: Load Crosswalk — SOC5 → OCC2010

Source: Lightcast 2021 SOC5 to OCC2010 Crosswalk.
- Maps each Lightcast SOC5 code to a Census OCC2010 code
- `merge_type`: One-to-One vs Many-to-One (multiple SOC5 → same OCC2010)
- `weight`: fractional weight for many-to-one cases — used when aggregating postings up to OCC2010 level

In [6]:
xwalk = pd.read_excel(data_path + 'Lightcast 2021 SOC5 to OCC2010 Crosswalk.xlsx', sheet_name='Sheet 1')
xwalk.columns = xwalk.columns.str.lower()

print(f'Crosswalk rows: {len(xwalk):,}')
print(f'Unique SOC5:    {xwalk["soc5"].nunique()}')
print(f'Unique OCC2010: {xwalk["occ2010"].nunique()}')
print()
print('Merge type breakdown:')
print(xwalk['merge_type'].value_counts())
xwalk.head()

Crosswalk rows: 799
Unique SOC5:    796
Unique OCC2010: 422

Merge type breakdown:
merge_type
One-to-One    794
Split           5
Name: count, dtype: int64


,soc5,soc5_title,occ2010,occ2010_title,merge_type,weight,jtrd
0,11-1011,Chief Executives,10,chief executives and legislators/public admini...,One-to-One,1.0,1
1,11-1021,General and Operations Managers,20,general and operations managers,One-to-One,1.0,1
2,11-1031,Legislators,10,chief executives and legislators/public admini...,One-to-One,1.0,1
3,11-2011,Advertising and Promotions Managers,30,"managers in marketing, advertising, and public...",One-to-One,1.0,1
4,11-2021,Marketing Managers,30,"managers in marketing, advertising, and public...",One-to-One,1.0,1


---
## Step 3: Load Portability Indices

Two versions from Jacob:
- **v1** (`portability_by_occupation.csv`): `portability` score, keyed by Census `occ`
- **v2** (`portability_index_fixed_delta1.csv`): `portability_index` (min-max normalized), keyed by Census `occ`

Both use Census occupation codes — same key as OCC2010 in the crosswalk and OCC in the CPS.

In [7]:
port_v1 = pd.read_csv(data_path + 'portability_by_occupation.csv')
port_v2 = pd.read_csv(data_path + 'portability_index_fixed_delta1.csv')

print('V1 columns:', port_v1.columns.tolist())
print(f'V1 rows: {len(port_v1):,}')
print()
print('V2 columns:', port_v2.columns.tolist())
print(f'V2 rows: {len(port_v2):,}')

# keep only the columns we need
port_v1 = port_v1[['occ', 'occ_title', 'portability']].rename(columns={'portability': 'portability_v1'})
port_v2 = port_v2[['occ', 'occ_title', 'portability_index']].rename(columns={'portability_index': 'portability_v2'})

print()
print('V1 portability distribution:')
print(port_v1['portability_v1'].describe().round(2))
print()
print('V2 portability distribution:')
print(port_v2['portability_v2'].describe().round(2))

V1 columns: ['occ', 'portability', 'weighted_employment', 'portability_per_million', 'occ_title', 'rank', 'percentile', 'rank_per_worker', 'percentile_per_worker']
V1 rows: 525

V2 columns: ['occ', 'port_rate', 'rank', 'portability_index', 'portability_minmax', 'occ_title', 'weighted_employment']
V2 rows: 525

V1 portability distribution:
count    525.00
mean       0.48
std        0.98
min        0.00
25%        0.05
50%        0.17
75%        0.47
max        8.89
Name: portability_v1, dtype: float64

V2 portability distribution:
count    525.00
mean       0.50
std        0.29
min        0.00
25%        0.25
50%        0.50
75%        0.75
max        1.00
Name: portability_v2, dtype: float64


---
## Step 4: Pull Job Postings 12-Month % Change from Lightcast

Query monthly postings by SOC5, 2019–2025. 2019 is needed to compute the 12-month change for Jan 2020.
Then:
1. Compute 12-month % change within each SOC5
2. Crosswalk SOC5 → OCC2010 using the Lightcast crosswalk weights
3. Aggregate to OCC2010 × year (annual average of monthly 12m changes)

In [8]:
query_post = '''
select
    date_trunc('month', p.posted)::date as year_month,
    year(p.posted)                      as year,
    month(p.posted)                     as month,
    p.soc_2021_5,
    p.soc_2021_5_name,
    count(distinct p.id)                as total_postings
from emsi.us.postings p
where year(p.posted) between 2019 and 2025
group by all
order by p.soc_2021_5, year_month
'''

cur_post = conn.cursor().execute(query_post)
post_monthly = pd.DataFrame.from_records(
    iter(cur_post),
    columns=[x[0] for x in cur_post.description]
)
post_monthly.columns = post_monthly.columns.str.lower()
post_monthly['year_month'] = pd.to_datetime(post_monthly['year_month'])
print(f'Rows: {len(post_monthly):,}  |  SOC5 codes: {post_monthly["soc_2021_5"].nunique()}')
post_monthly.head()

Rows: 63,644  |  SOC5 codes: 768


,year_month,year,month,soc_2021_5,soc_2021_5_name,total_postings
0,2019-01-01,2019,1,11-1011,Chief Executives,4343
1,2019-02-01,2019,2,11-1011,Chief Executives,4461
2,2019-03-01,2019,3,11-1011,Chief Executives,4415
3,2019-04-01,2019,4,11-1011,Chief Executives,4031
4,2019-05-01,2019,5,11-1011,Chief Executives,5324


In [27]:
# compute 12-month % change within each SOC5
post_monthly = post_monthly.sort_values(['soc_2021_5', 'year_month'])
post_monthly['pct_change_12m'] = (
    post_monthly.groupby('soc_2021_5')['total_postings']
    .pct_change(periods=12) * 100
)

# keep 2020 onward (2019 was only needed for the 12m lag)
post_monthly = post_monthly[post_monthly['year'] >= 2020].copy()

print(f'Rows after filtering to 2020+: {len(post_monthly):,}')
print(f'NaN pct_change_12m: {post_monthly["pct_change_12m"].isna().sum():,}')
post_monthly[['soc_2021_5', 'year_month', 'total_postings', 'pct_change_12m']].head()

Rows after filtering to 2020+: 54,576
NaN pct_change_12m: 130


,soc_2021_5,year_month,total_postings,pct_change_12m
12,11-1011,2020-01-01,4949,13.953488
13,11-1011,2020-02-01,5035,12.867070
14,11-1011,2020-03-01,4884,10.622877
15,11-1011,2020-04-01,3199,-20.640040
16,11-1011,2020-05-01,3437,-35.443276


In [28]:
# crosswalk SOC5 → OCC2010, then aggregate to OCC2010 x year
post_xw = post_monthly.merge(
    xwalk[['soc5', 'occ2010', 'weight']],
    left_on='soc_2021_5', right_on='soc5',
    how='inner'
)

print(f'Rows after crosswalk merge: {len(post_xw):,}')
print(f'SOC5 unmatched: {post_monthly["soc_2021_5"].nunique() - post_xw["soc_2021_5"].nunique()}')

# weighted average pct_change_12m within OCC2010 x year_month
def wavg(g):
    d = g.dropna(subset=['pct_change_12m'])
    if len(d) == 0:
        return np.nan
    return np.average(d['pct_change_12m'], weights=d['weight'])

post_occ_month = (
    post_xw.groupby(['occ2010', 'year', 'year_month'])
    .apply(wavg)
    .reset_index(name='pct_change_12m')
)

# annual average across months within OCC2010 x year
post_occ_year = (
    post_occ_month.dropna(subset=['pct_change_12m'])
    .groupby(['occ2010', 'year'])['pct_change_12m']
    .mean()
    .reset_index(name='postings_change_12m')
)

print(f'\nOCC2010 x year cells: {len(post_occ_year):,}')
print(f'Unique OCC2010: {post_occ_year["occ2010"].nunique()}')
post_occ_year.head()

Rows after crosswalk merge: 54,648
SOC5 unmatched: 2

OCC2010 x year cells: 2,520
Unique OCC2010: 420


/var/folders/6h/l1_lq8b96y72mkslk3phrlpw0000gn/T/ipykernel_62024/838935222.py:20: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(wavg)


,occ2010,year,postings_change_12m
0,10,2020,-11.257475
1,10,2021,38.657211
2,10,2022,26.468131
3,10,2023,-8.595028
4,10,2024,-8.140620


---
## Step 5: Merge All Datasets

Merge key: **OCC2010 × Year**
- CPS weeks unemployed: `occ` × `year`
- Postings % change: `occ2010` × `year`
- Portability v1 and v2: `occ` (time-invariant, merges on occ only)

In [29]:
# merge CPS + postings on occ x year
df = cps_occ.merge(
    post_occ_year,
    left_on=['occ', 'year'],
    right_on=['occ2010', 'year'],
    how='inner'
).drop(columns='occ2010')

# merge portability (time-invariant)
df = df.merge(port_v1, on='occ', how='left')
df = df.merge(port_v2[['occ', 'portability_v2']], on='occ', how='left')

print(f'Final panel rows:       {len(df):,}')
print(f'Unique occupations:     {df["occ"].nunique()}')
print(f'Years:                  {sorted(df["year"].unique())}')
print(f'Missing portability v1: {df["portability_v1"].isna().sum()}')
print(f'Missing portability v2: {df["portability_v2"].isna().sum()}')
df.head()

Final panel rows:       1,761
Unique occupations:     327
Years:                  [2020, 2021, 2022, 2023, 2024, 2025]
Missing portability v1: 0
Missing portability v2: 0


,occ,year,wks_unem_mean,postings_change_12m,occ_title,portability_v1,portability_v2
0,10,2020,3.011996,-11.257475,Chief executives,1.306583,0.887405
1,10,2021,5.343740,38.657211,Chief executives,1.306583,0.887405
2,10,2022,4.640960,26.468131,Chief executives,1.306583,0.887405
3,10,2023,2.032711,-8.595028,Chief executives,1.306583,0.887405
4,10,2024,2.919432,-8.140620,Chief executives,1.306583,0.887405


In [30]:
df[['wks_unem_mean', 'postings_change_12m', 'portability_v1', 'portability_v2']].describe().round(2)

,wks_unem_mean,postings_change_12m,portability_v1,portability_v2
count,1761.00,1761.00,1761.00,1761.00
mean,5.59,14.55,0.56,0.52
std,6.24,43.91,1.10,0.29
min,0.00,-72.34,0.00,0.00
25%,0.90,-8.60,0.07,0.28
50%,4.21,5.24,0.19,0.52
75%,7.76,24.99,0.58,0.78
max,48.00,683.51,8.89,1.00


---
## Step 6: Regression — Equation 7

Three specifications:
1. **Baseline** — no fixed effects
2. **With occupation + year FE** — absorbs time-invariant occ characteristics and common year shocks
3. **Excl. 2020–2022** — robustness check excluding COVID years

In [35]:
reg_df = df.dropna(subset=['wks_unem_mean', 'postings_change_12m', 'portability_v2']).copy()
reg_df['occ_str']  = reg_df['occ'].astype(str)
reg_df['year_str'] = reg_df['year'].astype(str)

# (1) baseline — no FE
m1 = smf.ols(
    'wks_unem_mean ~ postings_change_12m + portability_v2',
    data=reg_df
).fit(cov_type='HC1')

# (2) occupation + year FE
m2 = smf.ols(
    'wks_unem_mean ~ postings_change_12m + portability_v2 + C(occ_str) + C(year_str)',
    data=reg_df
).fit(cov_type='HC1')

# (3) excl. COVID years
reg_df_excl = reg_df[~reg_df['year'].between(2020, 2022)]
m3 = smf.ols(
    'wks_unem_mean ~ postings_change_12m + portability_v2 + C(occ_str) + C(year_str)',
    data=reg_df_excl
).fit(cov_type='HC1')

vars_of_interest = ['postings_change_12m', 'portability_v2']

print('=== (1) Baseline ===')
print(m1.summary().tables[1])
print(f'N={int(m1.nobs)}, R²={m1.rsquared:.3f}')

print('\n=== (2) Occ + Year FE ===')
print(m2.summary2().tables[1].loc[vars_of_interest])
print(f'N={int(m2.nobs)}, R²={m2.rsquared:.3f}')

print('\n=== (3) Excl. COVID (2020–2022), Occ + Year FE ===')
print(m3.summary2().tables[1].loc[vars_of_interest])
print(f'N={int(m3.nobs)}, R²={m3.rsquared:.3f}')

=== (1) Baseline ===
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
Intercept               5.1372      0.363     14.153      0.000       4.426       5.849
postings_change_12m     0.0133      0.004      3.285      0.001       0.005       0.021
portability_v2          0.4871      0.507      0.960      0.337      -0.507       1.482
N=1761, R²=0.009

=== (2) Occ + Year FE ===
                        Coef.  Std.Err.         z     P>|z|    [0.025  \
postings_change_12m  0.003981  0.004621  0.861466  0.388982 -0.005077   
portability_v2      -1.993744  0.991677 -2.010476  0.044381 -3.937395   

                       0.975]  
postings_change_12m  0.013039  
portability_v2      -0.050092  
N=1761, R²=0.257

=== (3) Excl. COVID (2020–2022), Occ + Year FE ===
                        Coef.  Std.Err.         z     P>|z|    [0.025  \
postings_change_12m -0.003834  0.0096

/opt/anaconda3/lib/python3.12/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 333, but rank is 327
  warnings.warn('covariance of constraints does not have full '
/opt/anaconda3/lib/python3.12/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 322, but rank is 305
  warnings.warn('covariance of constraints does not have full '


In [36]:
# robustness: portability v1
reg_df_v1 = df.dropna(subset=['wks_unem_mean', 'postings_change_12m', 'portability_v1']).copy()
reg_df_v1['occ_str']  = reg_df_v1['occ'].astype(str)
reg_df_v1['year_str'] = reg_df_v1['year'].astype(str)

m4 = smf.ols(
    'wks_unem_mean ~ postings_change_12m + portability_v1 + C(occ_str) + C(year_str)',
    data=reg_df_v1
).fit(cov_type='HC1')

print('=== (4) Portability V1 (robustness), Occ + Year FE ===')
vars_v1 = ['postings_change_12m', 'portability_v1']
print(m4.summary2().tables[1].loc[vars_v1])
print(f'N={int(m4.nobs)}, R²={m4.rsquared:.3f}')

=== (4) Portability V1 (robustness), Occ + Year FE ===
                        Coef.  Std.Err.         z     P>|z|    [0.025  \
postings_change_12m  0.003981  0.004621  0.861466  0.388982 -0.005077   
portability_v1      -0.606036  0.286589 -2.114655  0.034459 -1.167739   

                       0.975]  
postings_change_12m  0.013039  
portability_v1      -0.044333  
N=1761, R²=0.257


/opt/anaconda3/lib/python3.12/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 333, but rank is 327
  warnings.warn('covariance of constraints does not have full '


## for occupations that experienced large downturns:

In [37]:
# (4b) large-downturn subsample: bottom 10% of postings change
p10 = reg_df['postings_change_12m'].quantile(0.10)
print(f'Bottom 10% threshold: {p10:.1f}%')

reg_df_down = reg_df[reg_df['postings_change_12m'] <= p10].copy()
print(f'N in downturn subsample: {len(reg_df_down):,}')

m_down = smf.ols(
    'wks_unem_mean ~ postings_change_12m + portability_v2 + C(occ_str) + C(year_str)',
    data=reg_df_down
).fit(cov_type='HC1')

print('\n=== (4b) Large Downturns Only (bottom 10% postings change), Occ + Year FE ===')
print(m_down.summary2().tables[1].loc[vars_of_interest])
print(f'N={int(m_down.nobs)}, R²={m_down.rsquared:.3f}')

Bottom 10% threshold: -19.3%
N in downturn subsample: 177

=== (4b) Large Downturns Only (bottom 10% postings change), Occ + Year FE ===
                        Coef.  Std.Err.         z     P>|z|    [0.025  \
postings_change_12m  0.048746  0.149725  0.325573  0.744747 -0.244708   
portability_v2       3.176613  1.260238  2.520645  0.011714  0.706592   

                       0.975]  
postings_change_12m  0.342201  
portability_v2       5.646634  
N=177, R²=0.888


/opt/anaconda3/lib/python3.12/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 149, but rank is 36
  warnings.warn('covariance of constraints does not have full '


---
## Step 7: Robustness — Employment Level Changes (BLS OEWS)

As noted in the modelling plan, an alternative to job postings % change is the **12-month % change in employment level** from BLS OEWS. This is an annual measure so it matches the CPS year naturally.

In [40]:
import glob

oews_path = f"/Users/{user}/Documents/USF/ECON696/updated code/data/"
files = sorted(glob.glob(oews_path + 'national_M20*_dl.xlsx'))

bls_list = []
for f in files:
    year = int(f.split('national_M')[1][:4])
    tmp = pd.read_excel(f)
    tmp.columns = tmp.columns.str.lower()
    tmp = tmp[tmp['o_group'] == 'detailed'][['occ_code', 'occ_title', 'tot_emp']].copy()
    tmp['year'] = year
    bls_list.append(tmp)

bls = pd.concat(bls_list, ignore_index=True)
bls['tot_emp'] = pd.to_numeric(bls['tot_emp'], errors='coerce')
bls = bls.dropna(subset=['tot_emp']).sort_values(['occ_code', 'year'])
bls['emp_yoy'] = bls.groupby('occ_code')['tot_emp'].pct_change() * 100

print(f'SOC codes: {bls["occ_code"].nunique()}  |  Years: {sorted(bls["year"].unique())}')
bls.head()

SOC codes: 861  |  Years: [2019, 2020, 2021, 2022, 2023, 2024]


,occ_code,occ_title,tot_emp,year,emp_yoy
0,11-1011,Chief Executives,205890.0,2019,NaN
789,11-1011,Chief Executives,202360.0,2020,-1.714508
1578,11-1011,Chief Executives,200480.0,2021,-0.929037
2409,11-1011,Chief Executives,199240.0,2022,-0.618516
3239,11-1011,Chief Executives,211230.0,2023,6.017868


In [41]:
# crosswalk BLS SOC → OCC2010, merge into panel
bls_xw = bls.merge(
    xwalk[['soc5', 'occ2010', 'weight']],
    left_on='occ_code', right_on='soc5',
    how='inner'
)

bls_occ = (
    bls_xw.dropna(subset=['emp_yoy'])
    .groupby(['occ2010', 'year'])
    .apply(lambda g: np.average(g['emp_yoy'], weights=g['weight']))
    .reset_index(name='emp_change_12m')
)

df_bls = df.merge(
    bls_occ,
    left_on=['occ', 'year'],
    right_on=['occ2010', 'year'],
    how='inner'
).drop(columns='occ2010')

print(f'Panel rows with BLS emp change: {len(df_bls):,}')
df_bls[['occ', 'year', 'wks_unem_mean', 'emp_change_12m', 'portability_v1']].head()

Panel rows with BLS emp change: 1,434


/var/folders/6h/l1_lq8b96y72mkslk3phrlpw0000gn/T/ipykernel_62024/3563073782.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: np.average(g['emp_yoy'], weights=g['weight']))


,occ,year,wks_unem_mean,emp_change_12m,portability_v1
0,10,2020,3.011996,-1.804079,1.306583
1,10,2021,5.343740,-6.996006,1.306583
2,10,2022,4.640960,-2.215515,1.306583
3,10,2023,2.032711,-9.150077,1.306583
4,10,2024,2.919432,-9.018367,1.306583


In [42]:
# regression using BLS employment change instead of postings change
reg_bls = df_bls.dropna(subset=['wks_unem_mean', 'emp_change_12m', 'portability_v2']).copy()
reg_bls['occ_str']  = reg_bls['occ'].astype(str)
reg_bls['year_str'] = reg_bls['year'].astype(str)

m5 = smf.ols(
    'wks_unem_mean ~ emp_change_12m + portability_v2 + C(occ_str) + C(year_str)',
    data=reg_bls
).fit(cov_type='HC1')

print('=== (5) BLS Employment Change (vs Postings), Occ + Year FE ===')
print(m5.summary2().tables[1].loc[['emp_change_12m', 'portability_v2']])
print(f'N={int(m5.nobs)}, R²={m5.rsquared:.3f}')

=== (5) BLS Employment Change (vs Postings), Occ + Year FE ===
                   Coef.  Std.Err.         z         P>|z|    [0.025    0.975]
emp_change_12m -0.012029  0.031171 -0.385910  6.995632e-01 -0.073124  0.049066
portability_v2 -3.153584  0.548410 -5.750416  8.902404e-09 -4.228447 -2.078721
N=1434, R²=0.292


/opt/anaconda3/lib/python3.12/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 325, but rank is 316
  warnings.warn('covariance of constraints does not have full '


In [43]:
from IPython.display import display, HTML

def model_html(model, name, vars_show):
    t = model.summary2().tables[1].loc[vars_show][['Coef.', 'Std.Err.', 'P>|z|']]
    t = t.rename(columns={'Coef.': 'Coef', 'Std.Err.': 'SE', 'P>|z|': 'p-value'})
    t['Coef']    = t['Coef'].map('{:.4f}'.format)
    t['SE']      = t['SE'].map('{:.4f}'.format)
    t['p-value'] = t['p-value'].map('{:.3f}'.format)
    html = f'<h4>{name} &nbsp; <small>N={int(model.nobs):,} &nbsp; R²={model.rsquared:.3f}</small></h4>'
    html += t.to_html(border=1)
    return html

vars_main = ['postings_change_12m', 'portability_v2']
vars_v1   = ['postings_change_12m', 'portability_v1']
vars_bls  = ['emp_change_12m', 'portability_v2']

html_out = ''.join([
    model_html(m1,     '(1) Baseline — no FE',                             vars_main),
    model_html(m2,     '(2) Occ + Year FE',                                vars_main),
    model_html(m3,     '(3) Excl. COVID 2020–2022, Occ + Year FE',         vars_main),
    model_html(m4,     '(4) Robustness: Portability V1, Occ + Year FE',    vars_v1),
    model_html(m_down, '(4b) Large Downturns (bottom 10%), Occ + Year FE', vars_main),
    model_html(m5,     '(5) BLS Employment Change, Occ + Year FE',         vars_bls),
])

display(HTML(html_out))

,Coef,SE,p-value
postings_change_12m,0.0133,0.0041,0.001
portability_v2,0.4871,0.5073,0.337
,Coef,SE,p-value
postings_change_12m,0.0040,0.0046,0.389
portability_v2,-1.9937,0.9917,0.044
,Coef,SE,p-value
postings_change_12m,-0.0038,0.0097,0.692
portability_v2,-1.5694,1.9774,0.427
,Coef,SE,p-value
postings_change_12m,0.0040,0.0046,0.389


---
## Results Summary

**Main finding (Models 2, 4, 5):** Higher portability is consistently associated with fewer weeks unemployed. The coefficient on `portability_v2` is −2.0 (p=0.044) in the full panel with occupation and year fixed effects, and strengthens to −3.15 (p<0.001) when using BLS employment change instead of job postings as the demand shock measure. This holds across both portability index specifications (v1 and v2), suggesting the result is not sensitive to how portability is measured.

**Postings / employment change:** Neither `postings_change_12m` nor `emp_change_12m` is statistically significant once fixed effects are included. The demand shock variable itself does not predict unemployment duration after absorbing occupation and year variation — portability is the more relevant predictor of how long workers remain unemployed.

**COVID exclusion (Model 3):** Both coefficients become insignificant with N=878. After FE absorption, there is insufficient within-occupation variation to identify the effects in the non-COVID subsample. This should be treated as inconclusive rather than a null result.

**Large downturns (Model 4b):** The coefficient on `portability_v2` flips to +3.18 (p=0.012) in the bottom-10% postings change subsample, but this result is unreliable — N=177 is small relative to the number of occupation dummies, and statsmodels flags a rank-deficient FE covariance matrix. This specification should not be interpreted.

**Headline specifications:** Models 2 and 5 are the primary results. The consistent negative coefficient on portability across both demand shock measures and both portability indices supports the interpretation that **skill portability buffers workers against prolonged unemployment when their occupation experiences a demand downturn**.

---
## Section 5: AI-Exposed Occupations with Low Skill Portability

Which occupations face the double burden of high AI exposure **and** low portability — meaning workers in these jobs are most at risk of displacement with limited ability to move to alternatives?

**Data:**
- AI exposure: Anthropic Economic Index (`observed_exposure`), keyed by SOC code
- Portability: Jacob's portability index v2 (min-max normalized 0–1), keyed by Census OCC
- Crosswalk: Lightcast SOC5 → OCC2010 to align both measures

**Selection:** Occupations in the **top 5% of observed AI exposure** AND **bottom 5% of portability**.

In [50]:
# load Anthropic Economic Index
anthro = pd.read_csv(data_path + 'Anthropic_Economic_Index.csv')
anthro.columns = anthro.columns.str.lower()
print(f'Anthropic index rows: {len(anthro):,}')
print(f'observed_exposure range: {anthro["observed_exposure"].min():.3f} – {anthro["observed_exposure"].max():.3f}')
anthro.head(3)

Anthropic index rows: 756
observed_exposure range: 0.000 – 0.745


,occ_code,title,observed_exposure
0,11-1011,Chief Executives,0.0333
1,11-1021,General and Operations Managers,0.1378
2,11-1031,Legislators,0.0000


In [51]:
# crosswalk SOC → OCC2010, then merge portability v2
anthro_xw = anthro.merge(
    xwalk[['soc5', 'occ2010', 'weight']],
    left_on='occ_code', right_on='soc5',
    how='inner'
)

# weighted average observed_exposure within OCC2010 (for many-to-one cases)
anthro_occ = (
    anthro_xw.groupby('occ2010')
    .apply(lambda g: pd.Series({
        'observed_exposure': np.average(g['observed_exposure'], weights=g['weight']),
        'title': g.loc[g['weight'].idxmax(), 'title']
    }))
    .reset_index()
)

# merge portability v2
anthro_port = anthro_occ.merge(
    port_v2[['occ', 'portability_v2']],
    left_on='occ2010', right_on='occ',
    how='inner'
).drop(columns='occ')

print(f'Occupations after crosswalk + portability merge: {len(anthro_port):,}')
anthro_port.head(3)

Occupations after crosswalk + portability merge: 317


/var/folders/6h/l1_lq8b96y72mkslk3phrlpw0000gn/T/ipykernel_62024/538477375.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


,occ2010,observed_exposure,title,portability_v2
0,10,0.01665,Chief Executives,0.887405
1,20,0.13780,General and Operations Managers,0.944656
2,110,0.15590,Computer and Information Systems Managers,0.820611


In [52]:
# top 5% AI exposure, sorted by AI exposure descending
p95_exposure = anthro_port['observed_exposure'].quantile(0.95)
print(f'Top 5% AI exposure threshold: {p95_exposure:.3f}')

at_risk = anthro_port[
    anthro_port['observed_exposure'] >= p95_exposure
].sort_values('observed_exposure', ascending=False).reset_index(drop=True)

print(f'Occupations in top 5% AI exposure: {len(at_risk)}')

display(HTML(
    at_risk[['title', 'observed_exposure', 'portability_v2']]
    .rename(columns={
        'title': 'Occupation',
        'observed_exposure': 'AI Exposure (Observed)',
        'portability_v2': 'Portability Index (v2)'
    })
    .style
    .format({'AI Exposure (Observed)': '{:.3f}', 'Portability Index (v2)': '{:.3f}'})
    .background_gradient(subset=['Portability Index (v2)'], cmap='RdYlGn', vmin=0, vmax=1)
    .set_caption('Top 5% AI-Exposed Occupations, Sorted by Highest AI Exposure')
    .to_html()
))

Top 5% AI exposure threshold: 0.387
Occupations in top 5% AI exposure: 16


,Occupation,AI Exposure (Observed),Portability Index (v2)
0,Computer Programmers,0.745,0.559
1,Customer Service Representatives,0.701,0.985
2,Data Entry Keyers,0.671,0.981
3,Statistical Assistants,0.510,0.294
4,Technical Writers,0.475,0.258
5,Desktop Publishers,0.464,0.689
6,Public Relations Specialists,0.453,0.710
7,"Office Clerks, General",0.450,0.998
8,"Sales Representatives, Wholesale and Manufacturing, Technical and Scientific Products",0.449,0.983
9,"Securities, Commodities, and Financial Services Sales Agents",0.441,0.758


---
### Section 5 Regression: Does Portability Buffer the Unemployment Effect of AI Exposure?

Adds interaction terms to test two things:
1. **Portability × postings change** — does higher portability reduce the unemployment impact of a demand downturn?
2. **AI exposure × postings change** — does higher AI exposure amplify the unemployment impact of a downturn?

Note: since portability and AI exposure are time-invariant, they are absorbed by occupation FEs. But the **interaction terms** with `postings_change_12m` are identified because postings change varies within occupation over time.

In [53]:
# merge AI exposure into main panel
df_s5 = df.merge(
    anthro_port[['occ2010', 'observed_exposure']],
    left_on='occ', right_on='occ2010',
    how='inner'
).drop(columns='occ2010')

print(f'Panel rows with AI exposure: {len(df_s5):,}')
print(f'Unique occupations: {df_s5["occ"].nunique()}')
df_s5[['occ', 'year', 'wks_unem_mean', 'postings_change_12m', 'portability_v2', 'observed_exposure']].head()

Panel rows with AI exposure: 1,703
Unique occupations: 317


,occ,year,wks_unem_mean,postings_change_12m,portability_v2,observed_exposure
0,10,2020,3.011996,-11.257475,0.887405,0.01665
1,10,2021,5.343740,38.657211,0.887405,0.01665
2,10,2022,4.640960,26.468131,0.887405,0.01665
3,10,2023,2.032711,-8.595028,0.887405,0.01665
4,10,2024,2.919432,-8.140620,0.887405,0.01665


In [54]:
reg_s5 = df_s5.dropna(subset=['wks_unem_mean', 'postings_change_12m', 'portability_v2', 'observed_exposure']).copy()
reg_s5['occ_str']  = reg_s5['occ'].astype(str)
reg_s5['year_str'] = reg_s5['year'].astype(str)

# (S5-1) portability moderates demand shock
ms5_1 = smf.ols(
    'wks_unem_mean ~ postings_change_12m * portability_v2 + C(occ_str) + C(year_str)',
    data=reg_s5
).fit(cov_type='HC1')

# (S5-2) AI exposure amplifies demand shock
ms5_2 = smf.ols(
    'wks_unem_mean ~ postings_change_12m * observed_exposure + C(occ_str) + C(year_str)',
    data=reg_s5
).fit(cov_type='HC1')

# (S5-3) both interactions together
ms5_3 = smf.ols(
    'wks_unem_mean ~ postings_change_12m * portability_v2 + postings_change_12m * observed_exposure + C(occ_str) + C(year_str)',
    data=reg_s5
).fit(cov_type='HC1')

vars_s5_1 = ['postings_change_12m', 'portability_v2', 'postings_change_12m:portability_v2']
vars_s5_2 = ['postings_change_12m', 'observed_exposure', 'postings_change_12m:observed_exposure']
vars_s5_3 = ['postings_change_12m', 'portability_v2', 'observed_exposure',
             'postings_change_12m:portability_v2', 'postings_change_12m:observed_exposure']

html_s5 = ''.join([
    model_html(ms5_1, '(S5-1) Portability × Postings Change, Occ + Year FE',          vars_s5_1),
    model_html(ms5_2, '(S5-2) AI Exposure × Postings Change, Occ + Year FE',           vars_s5_2),
    model_html(ms5_3, '(S5-3) Both Interactions, Occ + Year FE',                       vars_s5_3),
])
display(HTML(html_s5))

/opt/anaconda3/lib/python3.12/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 324, but rank is 318
  warnings.warn('covariance of constraints does not have full '
/opt/anaconda3/lib/python3.12/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 324, but rank is 318
  warnings.warn('covariance of constraints does not have full '
/opt/anaconda3/lib/python3.12/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 326, but rank is 319
  warnings.warn('covariance of constraints does not have full '


,Coef,SE,p-value
postings_change_12m,-0.0076,0.0105,0.466
portability_v2,-2.2842,1.0384,0.028
postings_change_12m:portability_v2,0.0243,0.0171,0.155
,Coef,SE,p-value
postings_change_12m,0.0039,0.0051,0.437
observed_exposure,4.1266,1.6792,0.014
postings_change_12m:observed_exposure,0.0004,0.0199,0.983
,Coef,SE,p-value
postings_change_12m,-0.0075,0.0106,0.478
portability_v2,-2.1285,0.9937,0.032
